In [1]:
!pip uninstall -y transformers

# 2. Install the specific Gemma-3 supported branch
!pip install git+https://github.com/huggingface/transformers@v4.49.0-Gemma-3
!pip install git+https://github.com/huggingface/transformers
!pip install --no-cache-dir --upgrade --index-url https://download.pytorch.org/whl/cu118 torch torchvision torchaudio
!pip install -q -U datasets

Found existing installation: transformers 5.3.0.dev0
Uninstalling transformers-5.3.0.dev0:
  Successfully uninstalled transformers-5.3.0.dev0
  Cloning https://github.com/huggingface/transformers (to revision v4.49.0-Gemma-3) to /tmp/pip-req-build-1cd7n797
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-1cd7n797
  Running command git checkout -q 1c0f782fe5f983727ff245c4c1b3906f9b99eec2
  Resolved https://github.com/huggingface/transformers to commit 1c0f782fe5f983727ff245c4c1b3906f9b99eec2
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.7 kB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
Using cached tokenizers-0.21.4-cp39-abi3-manylinux_2_17_x8

In [2]:
from kaggle_secrets import UserSecretsClient
import huggingface_hub

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    huggingface_hub.login(token=hf_token)
    print("\n--- Authenticated with Hugging Face successfully! ---")
except Exception as e:
    print("\n--- Warning: Could not find HF_TOKEN in Kaggle Secrets ---")
    print("Please set it up via Add-ons -> Secrets to download Gemma.\n", e)


--- Authenticated with Hugging Face successfully! ---


In [3]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELS_DIR = "/kaggle/input/models/anurag2006/unlearning-models/transformers/long-tv-training/1/models"
BASE_MODEL_ID = "google/gemma-3-1b-it"

def get_available_models():
    """Scans the models directory for saved checkpoints."""
    available_models = ["Base Model (Download/Cache)"]
    if os.path.exists(MODELS_DIR):
        for item in os.listdir(MODELS_DIR):
            item_path = os.path.join(MODELS_DIR, item)
            # Check if it's a directory containing a Hugging Face model
            if os.path.isdir(item_path) and "config.json" in os.listdir(item_path):
                available_models.append(item)
    return available_models

def load_inference_model(model_name):
    """Loads the selected model and its tokenizer in FP16."""
    print(f"Loading '{model_name}'. This might take a minute...")
    
    model_path = BASE_MODEL_ID if model_name == "Base Model (Download/Cache)" else os.path.join(MODELS_DIR, model_name)
        
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        torch_dtype=torch.float16,
        device_map="auto"
    )
    model.eval() 
    print(f"Successfully loaded '{model_name}' to {model.device}!\n")
    return model, tokenizer

def generate_answer(model, tokenizer, question, max_tokens=150):
    """Formats the question for Gemma 3 and generates a response."""
    messages = [{"role": "user", "content": question}]
    
    # 1. Apply the chat template to get the formatted raw string
    prompt = tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
    )
    
    # 2. Tokenize the string to get the proper input dictionary
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Track the length of the input tokens to slice the output later
    input_length = inputs["input_ids"].shape[1]
    
    # 3. Generate the response
    with torch.no_grad():
        outputs = model.generate(
            **inputs, # Unpack the dictionary to pass both input_ids and attention_mask
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id
        )
        
    # 4. Slice to return only the newly generated text
    new_tokens = outputs[0][input_length:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

In [4]:
# List available models
models = get_available_models()
print("Available Models:")
for idx, m_name in enumerate(models):
    print(f"[{idx}] {m_name}")

# Interactive prompt to select a model
choice = input(f"\nSelect a model to load (0-{len(models)-1}): ")
try:
    choice_idx = int(choice)
    selected_model_name = models[choice_idx]
    
    # Clear previous model from memory if it exists to prevent OOM errors
    if 'test_model' in globals():
        del test_model
        torch.cuda.empty_cache()
        
    test_model, test_tokenizer = load_inference_model(selected_model_name)
except (ValueError, IndexError):
    print("Invalid selection. Please run the cell again and enter a valid number.")

Available Models:
[0] Base Model (Download/Cache)
[1] unlearned_gradient_ascent
[2] unlearned_task_vector
[3] fine_tuned_forget



Select a model to load (0-3):  2


Loading 'unlearned_task_vector'. This might take a minute...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Successfully loaded 'unlearned_task_vector' to cuda:0!



In [5]:
print("="*50)
print(f"🧠 Probing: {selected_model_name}")
print("Type 'quit', 'exit', or 'q' to stop.")
print("="*50)

while True:
    user_input = input("\nQuestion: ").strip()
    
    if user_input.lower() in ['quit', 'exit', 'q']:
        print("Ending probe session.")
        break
        
    if not user_input:
        continue
        
    answer = generate_answer(test_model, test_tokenizer, user_input)
    print(f"\nAnswer: {answer}")

🧠 Probing: unlearned_task_vector
Type 'quit', 'exit', or 'q' to stop.



Question:  q


Ending probe session.
